In [ ]:
# ─── CELL 1: Install & Setup ───
import subprocess, sys

def install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(args))

print("Installing packages...")
# Note: removed fsspec pinning to avoid breaking Kaggle integrations
install("datasets==2.20.0", "huggingface_hub", "datasketch", "pandas", "pyarrow")

from huggingface_hub import HfApi, login

HARDCODED_HF_TOKEN = ""

hf_token = None
api = None

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("✓ Loaded token from Kaggle Secrets")
except Exception:
    print("⚠ Kaggle Secrets not found — using hardcoded token fallback")
    hf_token = HARDCODED_HF_TOKEN

try:
    login(token=hf_token, add_to_git_credential=False)
    api = HfApi(token=hf_token)
    print("✓ HF Login Success")
except Exception as e:
    print(f"✗ HF Login failed: {e}")
    api = None

import gc, json, time, os, re, hashlib
from itertools import islice
import pandas as pd
from datasets import load_dataset
from datasketch import MinHash, MinHashLSH
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm


In [ ]:
# ─── CELL 2: CONFIG ───
HF_CHECKPOINT_REPO = "ncncomplete/karnalm-checkpoints"
HF_DATA_REPO = "ncncomplete/karnalm-data"
LANGUAGE = "Hindi"
LANGUAGE_CODE = "hi"

if api:
    try:
        api.create_repo(HF_CHECKPOINT_REPO, repo_type="dataset", exist_ok=True, private=True)
        api.create_repo(HF_DATA_REPO, repo_type="dataset", exist_ok=True, private=True)
    except Exception as e:
        print(f"⚠ Could not create repos: {e}")

CONFIG = {
    "min_doc_length": 100,
    "max_doc_length": 50000,
    "script_ratio": 0.3,
    "minhash_threshold": 0.85,
    "minhash_num_perm": 64,
    "target_unique_docs": 30000000,
    "stream_chunk_size": 10000,
    "lsh_rebuild_every": 300000,
    "CHECKPOINT_PUSH_EVERY": 25000,
    "DOCS_PER_SHARD": 250000,
    "MAX_RUN_SECONDS": 39600
}

SOURCES = [
    {
        "key": "sangraha_verified",
        "dataset": "ai4bharat/sangraha",
        "data_dir": "verified/hin",
        "split": "train",
        "trust_remote_code": True
    },
    {
        "key": "sangraha_unverified",
        "dataset": "ai4bharat/sangraha",
        "data_dir": "unverified/hin",
        "split": "train",
        "trust_remote_code": True
    },
    {
        "key": "indiccorp_hi",
        "dataset": "ai4bharat/IndicCorpV2",
        "config": "indiccorp_v2",
        "split": "hin_Deva",  # ← FIXED: Uses exact split name
        "trust_remote_code": True,
        "text_field": "text",
        "extra_filter": "hi_lang"
    },
    {
        "key": "mc4_hi",
        "dataset": "mc4",
        "config": "hi",
        "split": "train",
        "extra_filter": "hi_60",
        "trust_remote_code": True
    },
    {
        "key": "oscar_hi",
        "dataset": "oscar-corpus/OSCAR-2301",
        "config": "hi",
        "split": "train",
        "trust_remote_code": True,
        "text_field": "content"
    },
    {
        "key": "cc100_hi",
        "dataset": "cc100",
        "config": "hi",
        "split": "train",
        "trust_remote_code": True
    },
    {
        "key": "culturax_hi",
        "dataset": "uonlp/CulturaX",
        "config": "hi",
        "split": "train",
        "trust_remote_code": True
    },
    {
        "key": "wikipedia_hi",
        "dataset": "wikimedia/wikipedia",
        "config": "20231101.hi",
        "split": "train",
        "trust_remote_code": True
    }
]


In [ ]:
# ─── CELL 3: Helper Functions ───
def strip_html(text):
    return re.sub(r'<[^>]+>', '', text)

def check_repetition(text):
    lines = text.split('\n')
    line_counts = {}
    for line in lines:
        line = line.strip()
        if not line: continue
        line_counts[line] = line_counts.get(line, 0) + 1
        if line_counts[line] > 5:
            return True
    return False

def passes_language_filters(text, ex, source_config):
    if source_config.get("extra_filter") == "hi_lang":
        if ex.get("language") != "hin" and ex.get("lang_id") != "hi":
            return False

    alpha_chars = sum(1 for c in text if c.isalpha())
    script_chars = sum(1 for c in text if '\u0900' <= c <= '\u097F')
    
    if alpha_chars > 0 and (script_chars / alpha_chars) < CONFIG["script_ratio"]:
        return False
        
    if source_config.get("extra_filter") == "hi_60":
        if len(text) > 0 and (script_chars / len(text)) < 0.60:
            return False
            
    return True

def is_valid_doc(text, ex, source_config):
    if not text: return False
    
    clean_text = strip_html(text)
    if len(clean_text) < CONFIG["min_doc_length"] or len(clean_text) > CONFIG["max_doc_length"]:
        return False
        
    if check_repetition(clean_text): 
        return False
        
    if not passes_language_filters(clean_text, ex, source_config):
        return False
    
    return True

exact_hashes = set()

def build_fresh_lsh():
    return MinHashLSH(threshold=CONFIG["minhash_threshold"], num_perm=CONFIG["minhash_num_perm"])

lsh = build_fresh_lsh()

def compute_minhash(text):
    m = MinHash(num_perm=CONFIG["minhash_num_perm"])
    t = text.lower()
    for j in range(max(1, len(t)-4)):
        m.update(t[j:j+5].encode('utf-8'))
    return m

def get_exact_hash(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def load_exact_hashes():
    global exact_hashes
    if not api: return
    try:
        path = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, repo_type="dataset",
                               filename=f"{LANGUAGE_CODE}_exact_hashes.txt", token=hf_token)
        with open(path, 'r') as f:
            exact_hashes = set(line.strip() for line in f if line.strip())
        print(f"✓ Loaded {len(exact_hashes):,} exact hashes from previous runs")
    except Exception:
        print("No previous exact hashes found. Starting fresh.")
        exact_hashes = set()

def save_exact_hashes():
    if not api: return
    local_path = f"/kaggle/working/{LANGUAGE_CODE}_exact_hashes.txt"
    try:
        with open(local_path, 'w') as f:
            f.write('\n'.join(exact_hashes))
        api.upload_file(path_or_fileobj=local_path,
                        path_in_repo=f"{LANGUAGE_CODE}_exact_hashes.txt",
                        repo_id=HF_CHECKPOINT_REPO, repo_type="dataset", token=hf_token)
        print(f"  ✓ Exact hashes saved ({len(exact_hashes):,} hashes)")
        os.remove(local_path)
    except Exception as e:
        print(f"  ⚠ Could not save exact hashes: {e}")


In [ ]:
# ─── CELL 4: Checkpoint System ───
class CheckpointManager:
    def __init__(self):
        self.filename = f"{LANGUAGE_CODE}_checkpoint.json"
        self.local_path = f"/kaggle/working/{self.filename}"
        self.state = {
            "completed_sources": [],
            "current_source": None,
            "docs_processed_in_current": 0,
            "total_unique_docs": 0,
            "total_chars": 0,
            "output_shards_pushed": [],
            "next_shard_index": 0,
            "docs_in_current_shard": 0
        }
        self.load_checkpoint()

    def load_checkpoint(self):
        if not api: return
        try:
            path = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, repo_type="dataset",
                                   filename=self.filename, token=hf_token)
            with open(path, 'r', encoding='utf-8') as f:
                loaded = json.load(f)
            # Merge loaded state, keeping new keys with defaults
            self.state.update(loaded)
            if "next_shard_index" not in loaded: self.state["next_shard_index"] = 0
            if "docs_in_current_shard" not in loaded: self.state["docs_in_current_shard"] = 0
            print(f"✓ Resumed checkpoint from HF: {self.filename}")
        except Exception:
            print("No existing checkpoint found on HF. Starting fresh.")

    def save_checkpoint(self):
        if not api: return
        with open(self.local_path, 'w', encoding='utf-8') as f:
            json.dump(self.state, f, indent=2)
        try:
            api.upload_file(path_or_fileobj=self.local_path,
                            path_in_repo=self.filename,
                            repo_id=HF_CHECKPOINT_REPO,
                            repo_type="dataset", token=hf_token)
            print(f"  ✓ Checkpoint JSON saved to HF")
        except Exception as e:
            print(f"  ⚠ Checkpoint upload failed: {e}")

    def push_numbered_shard(self, source_key, buffer_file):
        if not api or not os.path.exists(buffer_file): return False
        try:
            df = pd.read_json(buffer_file, lines=True)
            if df.empty: return False
            idx = self.state.get("next_shard_index", 0)
            local_pq = f"/kaggle/working/{LANGUAGE_CODE}_shard_{idx:04d}.parquet"
            df.to_parquet(local_pq, index=False)
            remote_path = f"{LANGUAGE_CODE}/{source_key}_shard_{idx:04d}.parquet"
            api.upload_file(path_or_fileobj=local_pq, path_in_repo=remote_path,
                            repo_id=HF_DATA_REPO, repo_type="dataset", token=hf_token)
            shard_name = f"{source_key}_shard_{idx:04d}"
            if shard_name not in self.state["output_shards_pushed"]:
                self.state["output_shards_pushed"].append(shard_name)
            self.state["next_shard_index"] = idx + 1
            self.state["docs_in_current_shard"] = 0
            os.remove(buffer_file)
            os.remove(local_pq)
            print(f"  ✓ Shard uploaded: {remote_path} ({len(df):,} docs) — local files deleted")
            return True
        except Exception as e:
            print(f"  ⚠ Shard push failed: {e}")
            return False


In [ ]:
# ─── CELL 5: Main Pipeline Loop ───
ckpt = CheckpointManager()
start_time = time.time()
lsh_doc_count = 0
skipped_filter = 0
skipped_dup = 0
out_f = None
global lsh

load_exact_hashes()

try:
    for source in SOURCES:
        s_key = source["key"]

        if s_key in ckpt.state["completed_sources"]:
            print(f"Skipping completed source: {s_key}")
            continue

        print(f"\n━━ Processing Source: {s_key} ━━")

        if ckpt.state["current_source"] != s_key:
            ckpt.state["current_source"] = s_key
            ckpt.state["docs_processed_in_current"] = 0
            ckpt.save_checkpoint()

        buffer_file = f"/kaggle/working/{LANGUAGE_CODE}_{s_key}_buffer.jsonl"
        out_f = open(buffer_file, 'a', encoding='utf-8')

        try:
            print(f"  Loading dataset: {source['dataset']}")
            kwargs = {"streaming": True, "split": source.get("split", "train")}
            if "config" in source: kwargs["name"] = source["config"]
            if "data_dir" in source: kwargs["data_dir"] = source["data_dir"]
            if source.get("trust_remote_code"): kwargs["trust_remote_code"] = True

            ds = load_dataset(source["dataset"], **kwargs)

            skip_n = ckpt.state["docs_processed_in_current"]
            if skip_n > 0:
                print(f"  Resuming {s_key} from doc {skip_n}...")
                ds_iter = islice(iter(ds), skip_n, None)
            else:
                ds_iter = iter(ds)

            limit = source.get("limit", None)
            chunk = []
            pbar = tqdm(desc=f"  {s_key}", unit="doc")

            exit_reason = "exhausted"
            for ex in ds_iter:
                elapsed = time.time() - start_time
                if elapsed > CONFIG["MAX_RUN_SECONDS"]:
                    exit_reason = "timeout"
                    print(f"\n⏰ Time limit ({CONFIG['MAX_RUN_SECONDS']/3600:.1f}h) reached. Graceful exit...")
                    break

                if limit and (ckpt.state["docs_processed_in_current"] >= limit):
                    exit_reason = "limit"
                    print(f"\n  Reached configured limit of {limit} for {s_key}.")
                    break
                
                if ckpt.state["total_unique_docs"] >= CONFIG["target_unique_docs"]:
                    exit_reason = "target"
                    print(f"\n  🎯 Target reached!")
                    break

                ckpt.state["docs_processed_in_current"] += 1
                pbar.update(1)

                if source.get("text_concat"):
                    fields = source["text_concat"]
                    text = "\n".join([str(ex.get(f, "")) for f in fields])
                else:
                    text = ex.get(source.get("text_field", "text"), "")

                if not isinstance(text, str): text = str(text)
                text = text.strip()

                if not is_valid_doc(text, ex, source):
                    skipped_filter += 1
                    continue

                h_exact = get_exact_hash(text)
                if h_exact in exact_hashes:
                    skipped_dup += 1
                    continue

                chunk.append((text, h_exact))

                if len(chunk) >= CONFIG["stream_chunk_size"]:
                    for doc_text, doc_hash in chunk:
                        if ckpt.state["total_unique_docs"] >= CONFIG["target_unique_docs"]: break
                        m = compute_minhash(doc_text)
                        if len(lsh.query(m)) > 0:
                            skipped_dup += 1
                            continue
                        lsh.insert(f"d_{ckpt.state['total_unique_docs']}", m)
                        exact_hashes.add(doc_hash)
                        doc_obj = {"text": doc_text, "source": s_key, "lang": LANGUAGE_CODE}
                        out_f.write(json.dumps(doc_obj, ensure_ascii=False) + '\n')
                        ckpt.state["total_unique_docs"] += 1
                        ckpt.state["total_chars"] += len(doc_text)
                        ckpt.state["docs_in_current_shard"] = ckpt.state.get("docs_in_current_shard", 0) + 1
                        lsh_doc_count += 1
                        if lsh_doc_count >= CONFIG["lsh_rebuild_every"]:
                            lsh = build_fresh_lsh()
                            lsh_doc_count = 0
                    chunk = []
                    out_f.flush()
                    pbar.set_postfix({"unique": f"{ckpt.state['total_unique_docs']:,}", "dups": f"{skipped_dup:,}"})

                if ckpt.state["docs_in_current_shard"] >= CONFIG["DOCS_PER_SHARD"]:
                    out_f.close()
                    ckpt.push_numbered_shard(s_key, buffer_file)
                    ckpt.save_checkpoint()
                    gc.collect()
                    out_f = open(buffer_file, 'w', encoding='utf-8')

                if ckpt.state["docs_processed_in_current"] % CONFIG["CHECKPOINT_PUSH_EVERY"] == 0:
                    out_f.flush()
                    ckpt.save_checkpoint()
                    elapsed = time.time() - start_time
                    est_tokens = ckpt.state['total_chars'] / 4.0
                    print(f"\n  [Checkpoint] {s_key} | Processed: {ckpt.state['docs_processed_in_current']:,} | "
                          f"Unique: {ckpt.state['total_unique_docs']:,} | Tokens: {est_tokens/1e9:.3f}B | "
                          f"Time: {elapsed/60:.1f}m")

            # Flush remaining chunk
            for doc_text, doc_hash in chunk:
                if ckpt.state["total_unique_docs"] >= CONFIG["target_unique_docs"]: break
                m = compute_minhash(doc_text)
                if len(lsh.query(m)) > 0:
                    skipped_dup += 1
                    continue
                lsh.insert(f"d_{ckpt.state['total_unique_docs']}", m)
                exact_hashes.add(doc_hash)
                doc_obj = {"text": doc_text, "source": s_key, "lang": LANGUAGE_CODE}
                out_f.write(json.dumps(doc_obj, ensure_ascii=False) + '\n')
                ckpt.state["total_unique_docs"] += 1
                ckpt.state["total_chars"] += len(doc_text)
                ckpt.state["docs_in_current_shard"] = ckpt.state.get("docs_in_current_shard", 0) + 1
                lsh_doc_count += 1

            chunk = []
            out_f.flush()
            pbar.close()
            out_f.close()

            if exit_reason == "exhausted" or exit_reason == "target":
                print(f"✓ Source completed: {s_key}")
                ckpt.push_numbered_shard(s_key, buffer_file)
                ckpt.state["completed_sources"].append(s_key)
                ckpt.state["current_source"] = None
                ckpt.state["docs_processed_in_current"] = 0
                ckpt.save_checkpoint()
                if exit_reason == "target": break
            elif exit_reason == "limit":
                print(f"  Hit limit for {s_key}, not marking complete — increase limit to resume")
            elif exit_reason == "timeout":
                print(f"  Timeout reached, saving state and exiting source loop")
                break

        except Exception as e:
            print(f"  ✗ Failed processing {s_key}: {e}")
            if out_f and not out_f.closed:
                out_f.close()
            continue

except KeyboardInterrupt:
    print("\n🛑 Interrupted by user.")
except Exception as e:
    print(f"\n💥 Unexpected pipeline error: {e}")
finally:
    print("\n--- Pipeline Run Ended ---")
    if getattr(ckpt, 'state', {}).get("current_source"):
        print("Pushing partial buffer and checkpoint before exiting...")
        current = ckpt.state["current_source"]
        buffer_file = f"/kaggle/working/{LANGUAGE_CODE}_{current}_buffer.jsonl"
        try:
            if out_f is not None and not out_f.closed:
                out_f.close()
        except Exception:
            pass
        ckpt.push_numbered_shard(current, buffer_file)
        ckpt.save_checkpoint()
    save_exact_hashes()
    print("Checkpoint + hashes saved. Resume by re-running this notebook.")


In [ ]:
# ─── CELL 6: Final Summary ───
print("="*50)
print("📊 RUN SUMMARY")
print(f"Total Unique Docs: {ckpt.state['total_unique_docs']:,}")
print(f"Total Characters:  {ckpt.state['total_chars']:,}")
print(f"Estimated Tokens:  {ckpt.state['total_chars'] / 4.0 / 1e9:.3f} Billion")
print(f"Completed Sources: {ckpt.state['completed_sources']}")
print(f"Output Shards:     {len(ckpt.state['output_shards_pushed'])}")
print("="*50)
